# Overview

The figures of the README, made with the code in `src/` and saved to `figures/`.
Survey numbers follow the thesis (1-12); `load_survey` takes the 0-based index, so thesis survey 2 is index 1.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import anomaly as an
import propellers as pr
import tieline as tl
from dataloader import load_survey, load_object_locations
from paths import FIGURE_DIR
from signal_processing_tools import FS, SENSORS, butter_lowpass_filter, select_sensor

FIGURE_DIR.mkdir(exist_ok=True)
DPI = 150
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep", 5)

objects = load_object_locations()
baseline = load_survey(1, 1.25)   # thesis survey 2: constant heading, 0.87 m AGL, 1.5 m/s

## Survey 2: track labelling and sensor positions

In [ ]:
x0, y0 = baseline["Longitude [m]"].min(), baseline["Latitude [m]"].min()
data = baseline.iloc[::5]
off = data[data["Track_number"] < 0]
on = data[data["Track_number"] > 0]
tab10 = sns.color_palette("tab10")
colors = [tab10[i % 10] for i in range(on["Track_number"].nunique())]

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(off["Longitude [m]"] - x0, off["Latitude [m]"] - y0, s=0.3, color="0.8", rasterized=True)
for color, (track, g) in zip(colors, on.groupby("Track_number")):
    ax.scatter(g["Longitude [m]"] - x0, g["Latitude [m]"] - y0, s=0.3, color=color, rasterized=True)
    center = g[g["Sensor ID"] == 3]
    mid = center.iloc[len(center) // 2]
    ax.annotate(str(track), (mid["Longitude [m]"] - x0, mid["Latitude [m]"] - y0), ha="center", va="center",
                fontsize=9, fontweight="bold", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, lw=1.2))
ax.scatter(objects["Longitude [m]"] - x0, objects["Latitude [m]"] - y0, marker="*", s=250, color="black",
           edgecolor="white", linewidth=0.8, zorder=5)

ax.scatter([], [], s=20, color="0.8", label="Off-track (turns, take-off, landing)")
ax.scatter([], [], s=20, color=colors[0], label="On-track, coloured per track (5 sensor lines each)")
ax.scatter([], [], marker="*", s=120, color="black", label="Buried iron spheres (0.5 m deep)")
ax.legend(loc="upper left", fontsize=10, framealpha=0.95)
ax.set_aspect("equal")
ax.set_xlabel("Easting [m]")
ax.set_ylabel("Northing [m]")
ax.set_title("Survey 2: tracks detected by the dataloader", loc="left")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "survey_tracks.png", dpi=DPI, bbox_inches="tight")
plt.show()

## Survey 5: stationary drone, propellers on

In [ ]:
stationary = load_survey(4, 1.25)   # thesis survey 5
sensor = 3
signal = select_sensor(stationary, sensor)["Total field [nT]"].values

fig, ax = plt.subplots(figsize=(11, 5.5))
_, _, _, im = ax.specgram(signal, NFFT=1024, Fs=FS, noverlap=0, mode="psd", cmap="magma", vmin=-50, vmax=30)
fig.colorbar(im, ax=ax, pad=0.01).set_label("Power spectral density [dB]")
ax.set_ylim(0, 100)
ax.set_yticks(np.arange(0, 101, 10))
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title(f"Survey 5 (drone stationary, propellers on), sensor {sensor}", loc="left")
ax.grid(False)

label = dict(color="white", fontsize=9, fontweight="bold", va="bottom",
             bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5, lw=0))
ax.text(30, 19.5, "18 Hz", **label)
ax.text(30, 37.5, "36 Hz (2 x 18 Hz)", **label)
ax.text(30, 58, "propeller bands 42-47 Hz and 50-57 Hz", **label)
ax.text(30, 73.5, "72 Hz (4 x 18 Hz)", **label)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "spectrogram_propellers_on.png", dpi=DPI, bbox_inches="tight")
plt.show()

## Survey 8 (1 m/s): propeller ridges in the synchrosqueezed STFT

In [ ]:
survey_index, sensor = 7, 3   # thesis survey 8
flight = load_survey(survey_index, pr.SURVEY_SPEED)
data = select_sensor(flight, sensor).iloc[slice(*pr.TRIM[survey_index])]
_, Tx, _, ssq_freqs, scales = pr.propeller_ssq(data["Total field [nT]"], n_fft=2048)
ridge_idxs = pr.propeller_ridges(Tx, scales, *pr.ridge_settings(survey_index, sensor))

fmin, fmax = pr.PROPELLER_BAND
band = (ssq_freqs >= fmin) & (ssq_freqs <= fmax)
block = 20                                           # average over 0.1 s blocks (the SSQ-STFT is sparse)
n = (Tx.shape[1] - 2 * pr.EDGE) // block * block
inner = slice(pr.EDGE, pr.EDGE + n)
magnitude = np.abs(Tx[band][:, inner]).reshape(band.sum(), -1, block).mean(axis=2)
db = 20 * np.log10(magnitude + 1e-12)
t = (pr.EDGE + block * np.arange(magnitude.shape[1]) + block / 2) / FS

fig, ax = plt.subplots(figsize=(11, 5.5))
pcm = ax.pcolormesh(t, ssq_freqs[band], db, shading="auto", cmap="turbo",
                    vmin=np.percentile(db, 40), vmax=np.percentile(db, 99.8), rasterized=True)
fig.colorbar(pcm, ax=ax, pad=0.01).set_label("Magnitude [dB]")
for i, ridge in enumerate(np.argsort(ridge_idxs.mean(axis=0)), start=1):
    ridge_freq = np.median(ssq_freqs[ridge_idxs[inner, ridge].astype(int)].reshape(-1, block), axis=1)
    ax.plot(t, ridge_freq, color="white", linewidth=1.2, label="Extracted ridges" if i == 1 else None)
ax.set_ylim(fmin, fmax)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title(f"Survey 8 (1 m/s), sensor {sensor}: SSQ-STFT of the 37-60 Hz band", loc="left")
ax.legend(loc="upper right", fontsize=9)
ax.grid(False)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "propeller_ridges.png", dpi=DPI, bbox_inches="tight")
plt.show()
del Tx

## Survey 2: response of the buried spheres

In [ ]:
_, nearest = an.match_nearest_samples(baseline, objects)
windows = an.extract_dipole_windows(baseline, nearest, window_size=1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, column, title in [(axes[0], "Total field [nT]", "Unfiltered"),
                          (axes[1], "Total field [nT]_2Hz", "2 Hz lowpass")]:
    for sensor, color in zip(SENSORS, palette):
        w = windows[windows["Sensor ID"] == sensor]
        t = (np.arange(len(w)) - w["Center_Idx"].iloc[0]) / FS
        ax.plot(t, w[column] - w[column].median(), color=color, linewidth=1, label=f"Sensor {sensor}")
    ax.set_title(title, loc="left")
    ax.set_xlabel("Time from closest approach [s]")
axes[0].set_ylabel("Total field - median [nT]")
axes[1].legend(loc="upper right", fontsize=9)
fig.suptitle("Survey 2: the two buried spheres seen by the five sensors", x=0.01, ha="left")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "anomaly_response.png", dpi=DPI, bbox_inches="tight")
plt.show()

## Survey 2: lowpass filtered and detrended total field

In [ ]:
lowpassed = baseline.copy()
lowpassed["Total field 2Hz"] = lowpassed.groupby("Sensor ID")["Total field [nT]"].transform(
    lambda x: butter_lowpass_filter(x, 2, FS))
detrended, _ = tl.detrend_tracks(lowpassed, column="Total field 2Hz")

x0, y0 = baseline["Longitude [m]"].min(), baseline["Latitude [m]"].min()
x, y = detrended["Longitude [m]"] - x0, detrended["Latitude [m]"] - y0
ox, oy = objects["Longitude [m]"] - x0, objects["Latitude [m]"] - y0
zoom = 6   # m around the spheres
norm = dict(cmap="magma", vmin=-15, vmax=45)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7.5), gridspec_kw={"width_ratios": [1, 1.15]})
ax1.scatter(x, y, c=detrended["detrended"], s=0.5, rasterized=True, **norm)
ax1.add_patch(plt.Rectangle((ox.mean() - zoom, oy.mean() - zoom), 2 * zoom, 2 * zoom, fill=False, ec="cyan", lw=1.5))
ax1.set_title("Whole survey", loc="left")

near = (abs(x - ox.mean()) < zoom) & (abs(y - oy.mean()) < zoom)
sc = ax2.scatter(x[near], y[near], c=detrended.loc[near, "detrended"], s=4, rasterized=True, **norm)
ax2.scatter(ox, oy, marker="x", s=60, color="cyan", linewidth=1.5, label="Buried iron spheres")
ax2.set_xlim(ox.mean() - zoom, ox.mean() + zoom)
ax2.set_ylim(oy.mean() - zoom, oy.mean() + zoom)
ax2.legend(loc="upper left", fontsize=10)
ax2.set_title("Zoom on the buried spheres", loc="left")
fig.colorbar(sc, ax=ax2, pad=0.02).set_label("Detrended total field [nT]")

for ax in (ax1, ax2):
    ax.set_aspect("equal")
    ax.set_xlabel("Easting [m]")
    ax.set_ylabel("Northing [m]")
fig.suptitle("Survey 2: 2 Hz lowpass, linear trend removed per track (all sensors, no interpolation)", x=0.01, ha="left")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "detrended_field.png", dpi=DPI, bbox_inches="tight")
plt.show()